In [2]:
import torch
import torch.nn as nn
import torch.functional as F
import numpy
import math

### Tokenization and Embeddings

Tokenization will be done based on a character-level tokenizer.

In [28]:
class CharacterTokenizer():
    def __init__(self, s: str) -> None:
        chars: list[str] = sorted(set(s))
        self.vocab_size: int = len(chars)
        
        self.forward_mapping: dict = {ch: i for i, ch in enumerate(chars)}
        self.backward_mapping: dict = {i: ch for i, ch in enumerate(chars)}


    def encode(self, s: str) -> list[int]:
        return [self.forward_mapping[ch] for ch in s]


    def decode(self, tokens: list[int]) -> str:
        return "".join([self.backward_mapping[t] for t in tokens])


In [29]:
text = "Hello! I am learning how to build a transformer from scratch using PyTorch."

tokenizer = CharacterTokenizer(text)

tokens = tokenizer.encode(text)

print(tokenizer.decode(tokens) == text)


True


The dimensions of the embeddings will be
$$
E_{sum}=E_{token}+E_{position}
$$

The shape of these tensors are as follows
- $E_{sum}$ : $x_1 + x_2 \in \mathbb{R^{T \times {d_{embed}}}}$
- $E_{token}$ and $E_{position}$ : $x \in \mathbb{R^{T \times {d_{embed}}}}$

where $T$ is the size of the sequence of tokens


In [30]:
tokens = torch.tensor(tokenizer.encode(text), dtype=torch.long)
d_embed = 64

embedding_matrix = torch.randn(tokenizer.vocab_size,d_embed)
pos_embedding_matrix = torch.randn(128, d_embed)
positions = torch.arange(len(tokens))

embeddings = embedding_matrix[tokens]
embedded_tokens = pos_embedding_matrix[positions] + embeddings

print(pos_embedding_matrix.shape)
print(embedded_tokens.shape)
embedded_tokens

torch.Size([128, 64])
torch.Size([75, 64])


tensor([[ 0.6644, -0.8703,  2.0174,  ...,  0.4949,  0.0789, -1.6922],
        [-0.5594, -0.7632,  1.5625,  ...,  3.2362, -0.9830, -1.3755],
        [ 0.1507,  1.7606, -0.0450,  ...,  0.4794,  0.2917, -1.9989],
        ...,
        [-0.0589, -0.7439,  1.1851,  ...,  0.6719,  0.2829, -0.9767],
        [ 1.7060,  0.7845,  2.8599,  ...,  0.4255, -1.1315,  1.9021],
        [ 0.4902,  3.1530,  2.1487,  ...,  1.4527,  2.4360,  0.2122]])

### Self-Attention Mechanism

We compute the self-attention via computing 3 representations: $W_q, W_k$ and $W_v$



Then, we calculate the scaled dot product and apply a causal mask. The formula is:
$$
S=\frac{QK^T}{\sqrt{d_k}}
$$

After obtaining this result, we apply the softmax multiply by the value vector
$$
Attention(Q, K, V)=Softmax(\frac{QK^T}{\sqrt{d_k}})V
$$

In [44]:
w_q, w_k, w_v = (torch.randn(64, 64), 
                 torch.randn(64, 64), 
                 torch.randn(64, 64))

query_vector = torch.matmul(embedded_tokens, w_q)
key_vector = torch.matmul(embedded_tokens, w_k)
value_vector = torch.matmul(embedded_tokens, w_v)

scaled_dot_product = torch.matmul(query_vector, key_vector.T) / math.sqrt(d_embed)

triu_rows, triu_cols = torch.triu_indices(row=scaled_dot_product.size(0), 
                                  col=scaled_dot_product.size(1),
                                  offset=1)

scaled_dot_product[triu_rows, triu_cols] = float("-inf")

scaled_dot_product.shape
soft_max = nn.Softmax(dim=1)

out = soft_max(scaled_dot_product)
print(out.sum(dim=1) == torch.ones(out.shape[0], out.shape[1]))

A = torch.matmul(out, value_vector)
print(f"Output Shape: {A.shape} \n{A}")



tensor([[True, True, True,  ..., True, True, True],
        [True, True, True,  ..., True, True, True],
        [True, True, True,  ..., True, True, True],
        ...,
        [True, True, True,  ..., True, True, True],
        [True, True, True,  ..., True, True, True],
        [True, True, True,  ..., True, True, True]])
Output Shape: torch.Size([75, 64]) 
tensor([[-14.7904,   7.3138,  -0.4234,  ..., -15.7378, -16.6609,  17.6201],
        [-16.8061,  -9.3532,   4.6717,  ...,   2.9402,   0.7871,  -2.2369],
        [-14.7904,   7.3138,  -0.4234,  ..., -15.7378, -16.6609,  17.6201],
        ...,
        [ 29.5400, -18.2805,   0.4996,  ..., -15.5649,   0.7371,  16.6819],
        [  1.1500, -10.0791,   5.6647,  ...,  -6.6102,  -3.8132,   4.8263],
        [ -2.2298,   3.8530,   4.0087,  ...,   6.1268, -11.0600,   9.6512]])


### Multi-Head Attention